# Compute the frequency spectrum of a timeseries

This notebook demonstrates how to compute the discrete Fourier Transform (FFT) of a timeseries using `temporal.fft`. The time dimension is detected automatically and the returned frequency coordinate is expressed in Hz (cycles per second). We use a short hourly 2m temperature timeseries, which has a strong diurnal (24-hour) cycle.

In [ ]:
import numpy as np

from earthkit import data as ekd
from earthkit import transforms as ekt

# Hourly 2m temperature timeseries for two locations (72 hourly steps, 3 days)
ds = ekd.from_source("sample", "era5-timeseries-multiple.nc").to_xarray()
da = ds["t2m"].sel(location="Reading")

da

## Compute the FFT

`temporal.fft` detects the time dimension (`valid_time`) automatically and returns a complex-valued result indexed by a `frequency` dimension. The frequency coordinate is derived from the hourly sampling and expressed in Hz.

In [ ]:
spectrum = ekt.temporal.fft(da)

spectrum

## Identify the dominant period

We keep the positive frequencies, compute the amplitude of each frequency component, and convert the frequency (Hz) to a period in hours. The largest amplitude corresponds to the dominant cycle in the data.

In [ ]:
# Keep positive frequencies only (drop the zero-frequency/mean component)
positive = spectrum.where(spectrum["frequency"] > 0, drop=True)
amplitude = np.abs(positive)

# Convert frequency in Hz to a period in hours
period_hours = 1.0 / (positive["frequency"] * 3600.0)
amplitude = amplitude.assign_coords(period_hours=("frequency", period_hours.data))

dominant_period = float(amplitude["period_hours"][amplitude.argmax("frequency")].item())
print(f"Dominant period: {dominant_period:.1f} hours")

In [ ]:
amplitude.plot.line(x="period_hours", marker="o")